#### Reference: minBPE Tokenization lecture by Andrej Karpathy (https://youtu.be/zduSFxRajkE?si=CxVSuSDkOlN-g8rl)

#### Building individual components

In [ ]:
# raw text
text = open("tokenizer_data/unicode_blogpost.txt", "r").read()
# text = open("tokenizer_data/taylor_swift.txt", "r").read()

# convert text to utf-8 encoding bytes
raw_bytes = text.encode("utf-8")

# convert bytes to their integer/decimal representation --> base vocab = int from 0 to 255
token_ids = list(raw_bytes)  # often verbosely written as list(map(int, raw_bytes))

print(f"text length = {len(text)} | raw bytes = {len(raw_bytes)} | total tokens = {len(token_ids)}")

In [ ]:
# get all the pairs of tokens
def get_pair_frequency(ids):
    counts = {}
    for id0, id1 in zip(ids, ids[1:]):
        counts[(id0, id1)] = 1 + counts.get((id0, id1), 0)
    return counts


# test the function
stats = get_pair_frequency(ids=token_ids)
print(stats)

In [ ]:
# get most frequently occurring pair of token IDs
pair = max(stats, key=lambda x: stats[x])

print(f"Pair {pair} occurs most frequently -- {stats[pair]} times")

In [ ]:
# merge the pair with a new token
# iterates through the entire sequence and replaces the "pair" of token ids with a newly minted token


def merge(ids, pair, new_token):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


# test function
print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

#### BPE Training, Encode + Decode

In [ ]:
text = open("tokenizer_data/unicode_blogpost.txt", "r").read()

# convert text to utf-8 encoding bytes
raw_bytes = text.encode("utf-8")

# convert bytes to their integer/decimal representation --> base vocab = int from 0 to 255
token_ids = list(raw_bytes)  # often verbosely written as list(map(int, raw_bytes))

print(f"text length = {len(text)} | raw bytes = {len(raw_bytes)} | total tokens = {len(token_ids)}")

In [ ]:
"""
BPE training

Input:
- training data (raw text) 
    - NOTE: raw text is converted to utf-8 encoded bytes and then to token IDs
- no. of merges to perform (num_merges) OR vocab_size

Output:
- merges dictionary --> maps token pair to new token
- vocab dictionary --> maps every token (base + new) to token ID
"""


def get_pair_frequency(ids):
    counts = {}
    for id0, id1 in zip(ids, ids[1:]):
        counts[(id0, id1)] = 1 + counts.get((id0, id1), 0)
    return counts


def merge(ids, pair, new_token):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


#### training parameters
num_merges = 20
vocab_size = 256 + num_merges

#### data
ids = token_ids.copy()

#### model
merges = {}  # maps merged pairs to new token
# base vocabulary map. Maps integer representation (ID) of bytes to bytes.
# mapping for new tokens added while training.
vocab = {idx: bytes([idx]) for idx in range(256)}

#### training
for i in range(num_merges):
    # get most frequently occurring token pair
    stats = get_pair_frequency(ids)
    top_pair = max(stats, key=lambda x: stats[x])

    # mint new token
    new_token = 255 + i

    # update the model
    merges[top_pair] = new_token
    vocab[new_token] = vocab[top_pair[0]] + vocab[top_pair[1]]
    print(f"Merging {top_pair} into a new token {new_token}")

    # replace pair with new token
    ids = merge(ids=ids, pair=top_pair, new_token=new_token)

#
print("---------------------------------------")
print(f"Original vocab size = 256 and sequence length = {len(token_ids)}")
print(f"Original vocab size = {vocab_size} and sequence length = {len(ids)}")
print(f"Compression ratio: {len(token_ids) / len(ids):.2f}X")


In [ ]:
# Encode -- Given raw text, obtain tokens
"""
Input:
- raw text
- merges{} of trained model

Functions reused:
- get_pair_frequency()
- merge()

Output:
- token_ids sequence
"""


def encode(text, merges):
    # text to bytes to token IDs
    token_ids = list(text.encode("utf-8"))

    while len(token_ids) >= 2:
        # get token pair frequency
        stats = get_pair_frequency(ids=token_ids)

        # find most eligible pair of tokens according to model's merges{}
        # eligible pair -- pair present in model's merges{} and sits before the other eligible pairs in merges{} order
        pair = min(stats, key=lambda x: merges.get(x, float("inf")))

        # no more pairs in our input data exist in model's merges{} --> stop merging
        if pair not in merges:
            break

        # pair exists in merges --> perform merge
        token = merges[pair]
        token_ids = merge(ids=token_ids, pair=pair, new_token=token)

    return token_ids


# test encoder using data different from training data
raw_text = open("tokenizer_data/taylor_swift.txt", "r").read()
tokens = encode(text=raw_text, merges=merges)


In [ ]:
# Decode -- Given token ids, return text
# reverse process: token IDs --> bytes --> text
"""
Input:
- token ids
- merges{} and vocab{} of trained model

Output:
- raw text
"""


def decode(token_ids, vocab):
    # token IDs --> bytes
    byte_seq = b"".join(vocab[x] for x in token_ids)

    # bytes --> text through utf-8 decoding
    text = byte_seq.decode("utf-8", errors="replace")  # "replace" replaces byte un-decodable by utf-8 to "?"
    return text


# test decoder
decoded_text = decode(token_ids=tokens, vocab=vocab)
print(decoded_text)

In [ ]:
# Check decoding encoded text returns same original text

decode(token_ids=encode(text=raw_text, merges=merges), vocab=vocab) == raw_text

#### Creating Tokenizer class

In [ ]:
# Tokenizer class


def get_pair_frequency(ids):
    counts = {}
    for id0, id1 in zip(ids, ids[1:]):
        counts[(id0, id1)] = 1 + counts.get((id0, id1), 0)
    return counts


def merge(ids, pair, new_token):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class BasicTokenizer:
    def __init__(self):
        # maps merged pairs to new token
        self.merges = {}
        # initialize base vocab. Maps integer representation (ID) of bytes to bytes.
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def train(self, text, vocab_size, verbose=False):
        # parameters
        assert vocab_size > 256
        num_merges = vocab_size - len(self.vocab)

        # data: text -> bytes -> token ids
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)

        for i in range(num_merges):
            # get most frequently occurring token pair
            stats = get_pair_frequency(ids=ids)
            top_pair = max(stats, key=lambda x: stats[x])

            # mint new token
            new_token = 256 + i

            # update the model
            self.merges[top_pair] = new_token
            self.vocab[new_token] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]

            if verbose:
                print(
                    f"merge {i + 1}/{num_merges}: {top_pair} -> {new_token} ({self.vocab[new_token]}) had {stats[top_pair]} occurrences"
                )

            # replace pair with new token
            ids = merge(ids=ids, pair=top_pair, new_token=new_token)

        if verbose:
            print("---------------------------------------")
            print(f"Original vocab size = 256 and sequence length = {len(text_bytes)}")
            print(f"New vocab size = {len(self.vocab)} and sequence length = {len(ids)}")
            print(f"Compression ratio: {len(text_bytes) / len(ids):.2f}X")

    def encode(self, text):
        # preprocess data: text -> bytes -> token ids
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)

        # use trained model to encode
        while len(ids) >= 2:
            # get token pair frequency
            stats = get_pair_frequency(ids=ids)

            # find most eligible pair
            # eligible pair -- pair present in model's merges{} and sits before the other eligible pairs in merges{} order
            pair = min(stats, key=lambda x: self.merges.get(x, float("inf")))

            # no more pairs in our input data exist in model's merges{} --> stop merging
            if pair not in self.merges:
                break

            # pair exists in merges --> perform merge
            token = self.merges[pair]
            ids = merge(ids=ids, pair=pair, new_token=token)
        return ids

    def decode(self, ids):
        # token IDs --> bytes
        byte_seq = b"".join(self.vocab[x] for x in ids)
        # bytes --> text through utf-8 decoding
        text = byte_seq.decode("utf-8", errors="replace")
        return text


In [ ]:
training_text = open("tokenizer_data/unicode_blogpost.txt", "r").read()
test_text = open("tokenizer_data/taylor_swift.txt", "r").read()

In [ ]:
# Train the tokenizer
tokenizer = BasicTokenizer()
tokenizer.train(text=training_text, vocab_size=276, verbose=True)

# Encode
test_token_ids = tokenizer.encode(text=test_text)
print("---------------------------------------")
print(f"Encoded test set. Total tokens = {len(test_token_ids)}")

# Decode
test_decoded_text = tokenizer.decode(ids=test_token_ids)
print("---------------------------------------")
print(f"Decoded text matches original text --> {test_decoded_text == test_text}")

#### Creating Regex Tokenizer class

In [1]:
import regex

GPT4_SPLIT_PATTERN = (
    r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
)
gpt4pat = regex.compile(GPT4_SPLIT_PATTERN)


# [update 1:] prev counts dictionary can be passed to the function now
def get_pair_frequency(ids, counts=None):
    counts = {} if counts is None else counts
    for id0, id1 in zip(ids, ids[1:]):
        counts[(id0, id1)] = 1 + counts.get((id0, id1), 0)
    return counts


def merge(ids, pair, new_token):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


class RegexTokenizer:
    def __init__(self):
        # maps merged pairs to new token
        self.merges = {}
        # initialize base vocab. Maps integer representation (ID) of bytes to bytes.
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def train(self, text, vocab_size, verbose=False):
        # parameters
        assert vocab_size > 256
        num_merges = vocab_size - len(self.vocab)

        # data: text -> chunks -> list of lists (bytes per chunk) --> list of lists (token ids per chunk)
        text_chunks = regex.findall(gpt4pat, text)
        ids = [list(chunk.encode("utf-8")) for chunk in text_chunks]  # list of lists; each sublist is for a chunk

        for i in range(num_merges):
            # [update 2:] stats is computed, and one top pair obtained from across all chunks
            stats = {}
            for chunk_ids in ids:
                get_pair_frequency(ids=chunk_ids, counts=stats)

            top_pair = max(stats, key=lambda x: stats[x])

            # mint new token
            new_token = 256 + i

            # update the model
            self.merges[top_pair] = new_token
            self.vocab[new_token] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]

            if verbose:
                print(
                    f"merge {i + 1}/{num_merges}: {top_pair} -> {new_token} ({self.vocab[new_token]}) had {stats[top_pair]} occurrences"
                )

            # [update 3:] same pair is replaced with same new token across all chunks
            ids = [merge(ids=chunk_ids, pair=top_pair, new_token=new_token) for chunk_ids in ids]  # list of lists

    def encode(self, text):
        # preprocess data: text -> bytes -> token ids
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)

        # use trained model to encode
        while len(ids) >= 2:
            # get token pair frequency
            stats = get_pair_frequency(ids=ids)

            # find most eligible pair
            # eligible pair -- pair present in model's merges{} and sits before the other eligible pairs in merges{} order
            pair = min(stats, key=lambda x: self.merges.get(x, float("inf")))

            # no more pairs in our input data exist in model's merges{} --> stop merging
            if pair not in self.merges:
                break

            # pair exists in merges --> perform merge
            token = self.merges[pair]
            ids = merge(ids=ids, pair=pair, new_token=token)
        return ids

    def decode(self, ids):
        # token IDs --> bytes
        byte_seq = b"".join(self.vocab[x] for x in ids)
        # bytes --> text through utf-8 decoding
        text = byte_seq.decode("utf-8", errors="replace")
        return text


In [2]:
training_text = open("tokenizer_data/unicode_blogpost.txt", "r").read()
test_text = open("tokenizer_data/taylor_swift.txt", "r").read()

In [4]:
# Train the tokenizer
tokenizer = RegexTokenizer()
tokenizer.train(text=training_text, vocab_size=276, verbose=True)

# Encode
test_token_ids = tokenizer.encode(text=test_text)
print("---------------------------------------")
print(f"Encoded test set. Total tokens = {len(test_token_ids)}")

# Decode
test_decoded_text = tokenizer.decode(ids=test_token_ids)
print("---------------------------------------")
print(f"Decoded text matches original text --> {test_decoded_text == test_text}")

merge 1/20: (105, 110) -> 256 (b'in') had 448 occurrences
merge 2/20: (32, 116) -> 257 (b' t') had 407 occurrences
merge 3/20: (32, 97) -> 258 (b' a') had 377 occurrences
merge 4/20: (101, 114) -> 259 (b'er') had 299 occurrences
merge 5/20: (99, 111) -> 260 (b'co') had 293 occurrences
merge 6/20: (226, 128) -> 261 (b'\xe2\x80') had 259 occurrences
merge 7/20: (257, 104) -> 262 (b' th') had 259 occurrences
merge 8/20: (32, 115) -> 263 (b' s') had 222 occurrences
merge 9/20: (32, 111) -> 264 (b' o') had 222 occurrences
merge 10/20: (100, 101) -> 265 (b'de') had 209 occurrences
merge 11/20: (114, 101) -> 266 (b're') had 207 occurrences
merge 12/20: (105, 116) -> 267 (b'it') had 176 occurrences
merge 13/20: (32, 260) -> 268 (b' co') had 171 occurrences
merge 14/20: (262, 101) -> 269 (b' the') had 168 occurrences
merge 15/20: (256, 103) -> 270 (b'ing') had 167 occurrences
merge 16/20: (97, 116) -> 271 (b'at') had 153 occurrences
merge 17/20: (101, 110) -> 272 (b'en') had 153 occurrences
mer